# Notebook 06 - Neural Networks

In Notebook 05 we built logistic regression. We took the weighted sum $z = w_0 + w_1 x_1 + w_2 x_2$ and pushed it through the sigmoid, which squashed that number into a probability between $0$ and $1$. Functions like sigmoid, which take a weighted sum and apply some *nonlinearity* to it (in the sense that the function is not a single straight line), are called **activation functions**.

In this notebook we first build the basic unit, a single **neuron**, and wire many of them together into a **neural network**, the kind of model behind most of modern machine learning. Once the structure is in place we take a closer look at various activation functions other than sigmoid.

## One neuron, then a whole network

Let us name the basic building block. A single **neuron** (also called a **node** or a **unit**) takes the inputs it is given, forms a weighted sum of them, and passes that sum through a *nonlinear* function $\psi$ called the **activation function**. Writing the inputs as $x_1, \dots, x_n$, the neuron computes
$$ z = b + w_1 x_1 + w_2 x_2 + \dots + w_n x_n, \qquad a = \psi(z), $$
where the $w_i$ are the weights, $b$ is the bias, $\psi$ is the activation function, and $a$ is the neuron's activation. 

>**Note 1:** Notice that a single neuron with a sigmoid activation is the logistic regression model we've previously seen in Notebook 05.

>**Note 2:** Also notice that if we use the **identity function** $\psi(z) = z$ as the activation function (though it's linear), then the neuron is the same as our linear regression model we've seen in Notebook 02.

![A single neuron forms a weighted sum of its inputs plus a bias, then passes it through the activation function](images/neuron.png)

We've seen that models like linear regression and logistic regression are essentially very limited in capability and we often need more complex models. The way to get more complex models is to wire many neurons together into a **neural network**, where the output of one neuron can be the input to another.

## The anatomy of a neural net, layer by layer

Neurons in a network are organized into **layers**, and the layers have names.

- The **input layer** is just the raw features we feed in, one slot per feature. For our diabetes data from Notebook 05 that is two slots, one for blood sugar and one for BMI. These slots do no computing of their own. They only hold the numbers and hand them forward.
- A **hidden layer** is a column of neurons sitting between the input and the output. Every neuron in it takes all the values from the previous layer, forms its own weighted sum, and applies the activation function $\psi$. It is called hidden because we never read its values directly. They are internal scratch work the network uses on the way to an answer. A network can have one hidden layer or many (often many).
- The **output layer** is the final neuron or neurons that produce the prediction. It gathers the hidden outputs into one last weighted sum. There may be more than one output neuron (e.g., one per class in a multi-class classification problem).

When a network stacks up many hidden layers we call it **deep**, which is where the term **deep learning** comes from.

![A simple neural network with two inputs, one hidden layer of four neurons, and a single output](images/neural_network.png)

Trace the flow from left to right. The two features $x_1$ and $x_2$ enter on the left. Each of the four hidden neurons forms its own weighted sum of those two inputs, adds a bias, and applies a nonlinearity $\psi(\cdot)$. Writing $w_{ij}$ for the weight on the arrow from input $x_i$ into hidden neuron $j$, that neuron computes the *pre-activations* $z_j$ and the activations $a_j$ as
$$ z_j = w_{1j}\, x_1 + w_{2j}\, x_2 + b_j, \qquad a_j = \psi(z_j). $$
The output neuron on the right then takes the four activations $a_1, \dots, a_4$, forms one more weighted sum with weights $u_1, \dots, u_4$ and bias $c$, and produces
$$ h = c + u_1 a_1 + u_2 a_2 + u_3 a_3 + u_4 a_4, \qquad y = \psi(h)$$

Every $w_{ij}, b_j, u_j$ and $c$ in this figure is a separate weight. This tiny network already has $8$ input weights, $4$ output weights, and $5$ biases to learn, and real networks have millions or even billions.

> **Note:** The above form of a neural network is commonly referred to as a **fully connected feedforward network**, or sometimes as **multilayer perceptron**. The "fully connected" part means that every neuron in one layer connects to every neuron in the next layer. The "feedforward" part means that the connections only go forward, never looping back. 

## The activation function

We built the neuron and the network around a nonlinear function $\psi$. The sigmoid from Notebook 05 is one natural choice, so let us start there and see how well it serves as an activation function. Remember how we trained with gradient descent. Every weight update is proportional to the slope of $\psi$, because that slope is one of the links in the chain rule we worked through in Notebook 05. So the size of that slope decides how big a step each weight can take. Let us plot the sigmoid next to its own slope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z    = np.linspace(-8, 8, 400)
sig  = 1 / (1 + np.exp(-z))
dsig = sig * (1 - sig)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.4))

axL.plot(z, sig, color="tab:red", lw=2.5, label="sigmoid")
axL.axhline(0, color="gray", lw=0.8, ls=":")
axL.set_title("The sigmoid flattens out at both ends")
axL.set_xlabel("z  (the weighted sum)")
axL.set_ylabel("sigmoid(z)")
axL.legend(fontsize=9)

axR.plot(z, dsig, color="tab:green", lw=2.5, label="slope of the sigmoid")
axR.axhline(0.25, color="gray", lw=0.8, ls=":")
axR.axvspan(-8, -4, color="gray", alpha=0.15)
axR.axvspan(4, 8, color="gray", alpha=0.15)
axR.text(-6, 0.10, "flat,\nslope ~ 0", ha="center", fontsize=9, color="dimgray")
axR.text(6, 0.10, "flat,\nslope ~ 0", ha="center", fontsize=9, color="dimgray")
axR.set_title("Its slope is tiny almost everywhere")
axR.set_xlabel("z  (the weighted sum)")
axR.set_ylabel("slope of the sigmoid")
axR.legend(fontsize=9)
plt.tight_layout()
plt.show()

The picture on the right is the worry. The slope of the sigmoid is never larger than $0.25$, and out in the shaded regions, where the weighted sum $z$ is even moderately large in size, the slope is almost exactly $0$. We say the sigmoid has **saturated** there. When a neuron sits in one of those flat zones its slope is nearly zero, so its weight update is nearly zero too, and the neuron barely learns. Stack several sigmoids on top of one another and these tiny slopes get multiplied together, shrinking the update toward nothing. This is the famous **vanishing gradient** problem, and it makes deep stacks of sigmoids very slow to train.

This is why people went looking for other activation functions. Below are the sigmoid and its three most common companions:

In [ ]:
z = np.linspace(-6, 6, 400)

sigmoid    = 1 / (1 + np.exp(-z))
tanh       = np.tanh(z)
relu       = np.maximum(0, z)
leaky_relu = np.where(z > 0, z, 0.1 * z)

panels = [
    ("Sigmoid",    sigmoid,    "range 0 to 1"),
    ("Tanh",       tanh,       "range -1 to 1"),
    ("ReLU",       relu,       "max(0, z)"),
    ("Leaky ReLU", leaky_relu, "max(0.1 z, z)"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (name, curve, note) in zip(axes.ravel(), panels):
    ax.plot(z, curve, color="tab:red", lw=2.5)
    ax.axhline(0, color="black", lw=0.6)
    ax.axvline(0, color="black", lw=0.6)
    ax.set_title(f"{name}    ({note})")
    ax.set_xlabel("z")
    ax.set_ylabel(f"{name}(z)")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

- **Tanh** (*hyperbolic tangent*) has the same gentle S shape as the sigmoid, but it runs from $-1$ to $1$ instead of $0$ to $1$. Being centered on zero often helps training, although it still saturates and goes flat at both ends just like the sigmoid.

$$ \tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}} $$

- **ReLU**, short for *rectified linear unit*, is one of the most common activation functions in modern networks. For any positive input its slope is exactly $1$, so it never saturates on that side and the gradient flows through nicely.

$$ \mathrm{ReLU}(z) = \max(0,\, z) $$

- **Leaky ReLU** patches a small weakness of ReLU. A ReLU neuron that gets stuck always receiving negative inputs outputs zero forever and stops learning, called a *dead neuron*. Leaky ReLU keeps a small slope on the negative side, so a little signal always leaks through and the neuron can recover.

$$ \mathrm{LeakyReLU}(z) = \max(\alpha z,\, z), \quad \alpha \text{ is a small positive constant like 0.1} $$

### Why $\psi$ has to be nonlinear

Notice that every activation function above has bends or flat spots. None of them is a straight line, and that is not an accident. Suppose we skipped $\psi$ altogether and let each neuron output its raw weighted sum. It is tempting to think the weighted sums are doing all the real work and $\psi$ is just decoration, but the opposite is true. Feeding one weighted sum into another just gives a weighted sum of a weighted sum, which is itself nothing more than a single weighted sum. No matter how many layers we stack, the whole network collapses back into one plain linear model, the kind that can only draw a straight line. The activation function is the one ingredient that breaks this collapse and lets extra layers actually buy us extra flexibility. 

To see this, suppose we have a network with one hidden layer of four neurons, as in the figure above, but no activation function. The output neuron computes

$$
y
=
c
+
u_1 z_1
+
u_2 z_2
+
u_3 z_3
+
u_4 z_4,
$$

where each $z_j$ is a weighted sum of the inputs $x_1$ and $x_2$. Substituting those sums in gives

$$
\begin{aligned}
y =\;& c
+ u_1 (b_1 + w_{11}x_1 + w_{21}x_2)
+ u_2 (b_2 + w_{12}x_1 + w_{22}x_2) \\
&+ u_3 (b_3 + w_{13}x_1 + w_{23}x_2)
+ u_4 (b_4 + w_{14}x_1 + w_{24}x_2).
\end{aligned}
$$

Collecting terms gives

$$
\begin{aligned}
y =\;&
c + u_1 b_1 + u_2 b_2 + u_3 b_3 + u_4 b_4 \\
&+ (u_1 w_{11} + u_2 w_{12} + u_3 w_{13} + u_4 w_{14})\,x_1 \\
&+ (u_1 w_{21} + u_2 w_{22} + u_3 w_{23} + u_4 w_{24})\,x_2.
\end{aligned}
$$

This is just a single weighted sum of the inputs, with bias

$$
c + u_1 b_1 + u_2 b_2 + u_3 b_3 + u_4 b_4,
$$

and weights

$$
u_1 w_{11} + u_2 w_{12} + u_3 w_{13} + u_4 w_{14},
$$

and

$$
u_1 w_{21} + u_2 w_{22} + u_3 w_{23} + u_4 w_{24},
$$

the same form as a single neuron with no hidden layer. The network is just a more complicated way to write a single weighted sum, and it can only learn to draw a straight line.

The plot below makes the point on a curvy little dataset, comparing a network with no activation against the same network using ReLU.

In [ ]:
from sklearn.neural_network import MLPRegressor

rng = np.random.default_rng(0)
x = np.linspace(-3, 3, 60)
y = np.sin(1.5 * x) + rng.normal(0, 0.1, x.shape)
X = x.reshape(-1, 1)

net_linear = MLPRegressor(hidden_layer_sizes=(30, 30), activation="identity",
                          max_iter=5000, random_state=0).fit(X, y)
net_relu   = MLPRegressor(hidden_layer_sizes=(30, 30), activation="relu",
                          max_iter=5000, random_state=0).fit(X, y)

grid = np.linspace(-3, 3, 300).reshape(-1, 1)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, net, title in [
    (axL, net_linear, "No activation: only a straight line"),
    (axR, net_relu,   "With ReLU activation: the same network bends"),
]:
    ax.scatter(x, y, color="black", s=18, alpha=0.7, label="data")
    ax.plot(grid, net.predict(grid), color="tab:red", lw=2.5, label="network output")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x")
    ax.legend(fontsize=9)
axL.set_ylabel("y")
plt.tight_layout()
plt.show()

On the left the network has plenty of layers and neurons, yet with no activation function it still manages only a straight line. On the right the same network, now with ReLU sitting between its layers, bends and follows the data.

## Back to the patients, with a curved boundary

Let us close the loop with the diabetes dataset from Notebook 05. There we noted that logistic regression can only draw one straight decision boundary, and that a more flexible model could bend around the data. A neural network is one such flexible model. Below we place plain logistic regression, which is a single neuron, next to a small network with two hidden layers, both trained on the same patients.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
n0 = 120
sugar0 = rng.normal(100, 15, n0)
bmi0   = rng.normal(26, 4.0, n0)
n1 = 120
sugar1 = rng.normal(132, 18, n1)
bmi1   = rng.normal(32, 5.0, n1)

sugar = np.concatenate([sugar0, sugar1])
bmi   = np.concatenate([bmi0, bmi1])
X = np.column_stack([sugar, bmi])
y = np.concatenate([np.zeros(n0), np.ones(n1)]).astype(int)

C    = ["#9ecae1", "#08519c"]
NAME = ["negative (0)", "positive (1)"]

logreg = LogisticRegression().fit(X, y)
net = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 16), activation="relu",
                  max_iter=4000, random_state=0),
).fit(X, y)

xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 10, X[:, 0].max() + 10, 300),
    np.linspace(X[:, 1].min() - 3,  X[:, 1].max() + 3,  300),
)
grid_pts = np.column_stack([xx.ravel(), yy.ravel()])

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.4), sharex=True, sharey=True)
panels = [
    (axL, logreg, "Logistic regression (one neuron): straight boundary"),
    (axR, net,    "Neural network (two hidden layers): curved boundary"),
]
for ax, mdl, title in panels:
    P = mdl.predict_proba(grid_pts)[:, 1].reshape(xx.shape)
    region = (P >= 0.5).astype(float)
    ax.contourf(xx, yy, region, levels=[-0.5, 0.5, 1.5], cmap="Blues", alpha=0.35)
    ax.contour(xx, yy, P, levels=[0.5], colors="black", linewidths=2)
    for c in (0, 1):
        m = y == c
        ax.scatter(X[m, 0], X[m, 1], color=C[c], edgecolor="black", s=28, linewidth=0.5,
                   label=NAME[c])
    acc = (mdl.predict(X) == y).mean()
    ax.set_title(f"{title}\n(accuracy {acc:.1%})", fontsize=10)
    ax.set_xlabel("blood sugar level")
axL.set_ylabel("BMI")
axL.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

The single neuron on the left commits to one straight cut, as it did in Notebook 05. The network on the right uses its hidden layers to bend the boundary around the cloud of patients and pick up a few more of them.

## Summary

- A **neuron** forms a weighted sum $z = b + w_1 x_1 + \dots + w_n x_n$ and applies an activation function $\psi$ to get its output $a = \psi(z)$.
- A **neural network** wires neurons into **layers**. The **input layer** holds the features, one or more **hidden layers** do the internal computing, and the **output layer** produces the prediction. Many hidden layers make the network **deep**, hence the termm **deep learning**.
- An **activation function** is the nonlinear function $\psi$ we apply to the weighted sum.
- Without a nonlinear $\psi$ a stack of neurons collapses into one plain linear model, so the nonlinearity is required in neural networks.
- Networks can bend decision boundaries into curves that a single neuron never could, but that same flexibility brings back the overfitting risk, so the lessons on regularization carry over.